In [2]:
pip install brainsmash

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
import nibabel as nib
from brainsmash.mapgen.sampled import Sampled

# 0.Processing

In [2]:
# =========================================================
# 0) 你要改的路径
# =========================================================
# 目标基因集表达矩阵：ROI × targetGenes（比如你筛选后的 top10 或你新的 gene set）
E_target_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_left_top10_top15_activate_kwx.csv"

# 全基因表达矩阵：ROI × AllGenes（你说你有这个）
E_all_path    = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_left_kwx.csv"

# FC / EEG 矩阵：ROI × ROI（你已经有，含 NaN OK）
FC_path       = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/eeg_left_top15_kwx.csv"

# atlas & label mapping：用来算 ROI centroid -> dist
atlas_path        = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/rawdata/MNI152NLin2009aSym_lausenne60.nii.gz"
label_table_path  = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/rawdata/lausanne_scale60_atlas_info.csv"   # 必须包含：roi_name列 + label_id列（见下面你要改列名）

out_dir = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/03.correlation/"
os.makedirs(out_dir, exist_ok=True)


# =========================================================
# 1) 你手动填入：你已经算好的真实相关 r_real
#    （代码不再重复算你的主结果）
# =========================================================
r_real = 0.3549   # <-- 你把这里换成你自己已经算好的那个 r

# 1.Utility function

In [3]:
# =========================================================
# 2) 工具函数：矩阵相关（上三角、忽略 NaN、不给0填充）
# =========================================================
def corr_upper_triangle(A: pd.DataFrame, B: pd.DataFrame, min_edges: int = 30):
    """
    返回 Pearson r，使用上三角（k=1），自动对齐 NaN mask。
    """
    if A.shape != B.shape:
        raise ValueError(f"Shape mismatch: A{A.shape} vs B{B.shape}")

    R = A.shape[0]
    iu = np.triu_indices(R, k=1)
    a = A.values[iu]
    b = B.values[iu]
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < min_edges:
        return np.nan, int(mask.sum())
    r = pearsonr(a[mask], b[mask])[0]
    return r, int(mask.sum())


def coexp_from_expr(E: pd.DataFrame, method: str = "pearson"):
    """
    ROI×gene -> ROI×ROI co-expression
    注意：这里是 ROI 之间的相关，所以要对 ROI 向量做相关：E.T.corr() 或 E.corr? 取决于 E 维度
    E = ROI×gene  => coexp = corr(ROI_i across genes) => E.T.corr()
    """
    return E.T.corr(method=method)

# 2.ROI Distance

In [4]:
# =========================================================
# 3) 计算 ROI 距离矩阵 dist（ROI×ROI）
#    - 用 atlas NIfTI 的 label 区域求 centroid（MNI 空间）
#    - 需要 ROI名 <-> label整数 的 mapping
# =========================================================
def compute_roi_centroids_from_atlas(atlas_path, label_table_path,
                                     roi_name_col="roi_name",
                                     label_id_col="label_id"):
    """
    返回：
      centroids_df: index=ROI name, columns=[x,y,z]
    """
    info = pd.read_csv(label_table_path)
    if roi_name_col not in info.columns or label_id_col not in info.columns:
        raise ValueError(
            f"label_table 缺列：需要 {roi_name_col} 和 {label_id_col}。\n"
            f"你现在的列有：{list(info.columns)}\n"
            f"请把函数参数 roi_name_col/label_id_col 改成你文件里的真实列名。"
        )

    atlas_img = nib.load(atlas_path)
    data = np.asanyarray(atlas_img.dataobj)
    affine = atlas_img.affine

    centroids = {}
    for _, row in info.iterrows():
        roi = str(row[roi_name_col]).strip()
        lab = int(row[label_id_col])

        vox = np.argwhere(data == lab)
        if vox.size == 0:
            continue

        # voxel -> world(MNI): [i,j,k,1] * affine^T
        vox_mean = vox.mean(axis=0)
        xyz = nib.affines.apply_affine(affine, vox_mean)

        centroids[roi] = xyz

    centroids_df = pd.DataFrame.from_dict(centroids, orient="index", columns=["x","y","z"])
    return centroids_df


def pairwise_euclidean_distance(coords: pd.DataFrame):
    """
    coords: index=ROI, columns=x,y,z
    return dist_df: ROI×ROI
    """
    X = coords[["x","y","z"]].values.astype(float)
    diff = X[:, None, :] - X[None, :, :]
    D = np.sqrt((diff ** 2).sum(axis=2))
    return pd.DataFrame(D, index=coords.index, columns=coords.index)

In [5]:
# =========================================================
# 4) 读取数据 + 对齐 ROI
# =========================================================
print("==== [LOAD] reading matrices ====")
E_target = pd.read_csv(E_target_path, index_col=0).apply(pd.to_numeric, errors="coerce")
E_all    = pd.read_csv(E_all_path, index_col=0).apply(pd.to_numeric, errors="coerce")
FC       = pd.read_csv(FC_path, index_col=0).apply(pd.to_numeric, errors="coerce")

E_target.index = E_target.index.astype(str).str.strip()
E_all.index    = E_all.index.astype(str).str.strip()
FC.index       = FC.index.astype(str).str.strip()
FC.columns     = FC.columns.astype(str).str.strip()

print("[INFO] E_target:", E_target.shape, " E_all:", E_all.shape, " FC:", FC.shape)

print("==== [DIST] compute ROI distance matrix ====")
# 你需要把下面两列名改成你 label_table 里真实的列名（只改这里就行）
centroids = compute_roi_centroids_from_atlas(
    atlas_path, label_table_path,
    roi_name_col="alias1",      # <-- 改成你表里的 ROI 名称列名
    label_id_col="id"       # <-- 改成你表里的 label 整数列名
)
print("[INFO] centroids computed:", centroids.shape)

# 对齐到你分析用的 ROI（以 FC 行列 & 表达矩阵共同 ROI 为准）
common_rois = (
    E_target.index
    .intersection(E_all.index)
    .intersection(FC.index)
    .intersection(FC.columns)
    .intersection(centroids.index)
)

print("[INFO] common_rois:", len(common_rois))
if len(common_rois) < 5:
    raise ValueError("共同 ROI 太少：请检查 ROI 命名是否一致（是否有 _1 后缀、是否左右脑前缀一致）。")

# 统一排序（保持一致顺序）
E_target = E_target.loc[common_rois, :]
E_all    = E_all.loc[common_rois, :]
FC       = FC.loc[common_rois, common_rois]
centroids = centroids.loc[common_rois, :]

dist = pairwise_euclidean_distance(centroids)
dist.to_csv(os.path.join(out_dir, "roi_distance_matrix_kwx.csv"))
print("[SAVE] roi_distance_matrix.csv  shape:", dist.shape)

==== [LOAD] reading matrices ====
[INFO] E_target: (10, 1563)  E_all: (64, 15634)  FC: (10, 10)
==== [DIST] compute ROI distance matrix ====
[INFO] centroids computed: (129, 3)
[INFO] common_rois: 10
[SAVE] roi_distance_matrix.csv  shape: (10, 10)


# 3.Control-1 空间自相关

In [12]:
# =========================================================
# 5) 控制实验 1：BrainSMASH（空间自相关 SA 控制） -> pSA
#    brainsmash==0.11.0 兼容版：Sampled 需要 index 参数
# =========================================================
print("\n==== [CTRL1] BrainSMASH SA-preserving null (small-ROI safe) ====")

n_perm = 10000
R, Gt = E_target.shape

D_np = dist.values
r_null_sa = np.full(n_perm, np.nan, dtype=float)

print("[INFO] ROI:", R, "targetGenes:", Gt, "n_perm:", n_perm)
print("[INFO] Note: ROI is small -> we will adapt knn/ns/nh per gene based on valid ROIs.")

for p in range(n_perm):
    E_surr = np.full((R, Gt), np.nan, dtype=float)

    for j, gene in enumerate(E_target.columns):
        x = E_target[gene].values.astype(float)

        mask = np.isfinite(x)
        n_valid = int(mask.sum())
        if n_valid < 5:
            continue

        x_valid = x[mask]
        D_sub = D_np[np.ix_(mask, mask)]

        # ---- adaptive parameters for small n_valid ----
        knn_sub = min(100, n_valid - 1)           # must be <= n_valid-1
        ns_sub  = min(8, n_valid)                 # sample size for variogram, must be <= n_valid
        ns_sub  = max(5, ns_sub)                  # keep it not too tiny
        nh_sub  = min(6, n_valid - 1)             # number of bins; keep small for stability
        nh_sub  = max(3, nh_sub)

        # index: (n_valid, knn_sub+1) nearest-neighbor order (including self)
        idx = np.argsort(D_sub, axis=1)[:, :knn_sub + 1].astype(np.int32)

        # brainsmash==0.11.0: Sampled(x, D, index, ns=..., nh=..., knn=...)
        gen = Sampled(x_valid, D_sub, idx, ns=ns_sub, nh=nh_sub, knn=knn_sub)
        xs_valid = gen(n=1).squeeze()

        x_s = np.full_like(x, np.nan, dtype=float)
        x_s[mask] = xs_valid
        E_surr[:, j] = x_s

    coexp_s = coexp_from_expr(
        pd.DataFrame(E_surr, index=common_rois, columns=E_target.columns),
        method="pearson"
    )
    r_s, n_edges = corr_upper_triangle(coexp_s, FC, min_edges=30)
    r_null_sa[p] = r_s

    if (p + 1) % 200 == 0:
        valid = np.isfinite(r_null_sa[:p+1]).sum()
        print(f"[CTRL1] {p+1}/{n_perm}  valid={valid}")

# pSA
r_null_sa = r_null_sa[np.isfinite(r_null_sa)]
p_sa = np.mean(r_null_sa >= r_real)

null_path = os.path.join(out_dir, "ctrl1_null_sa_brainsmash_kwx.csv")
sum_path  = os.path.join(out_dir, "ctrl1_summary_sa_kwx.csv")

pd.Series(r_null_sa, name="r_null_sa").to_csv(null_path, index=False)
pd.DataFrame({
    "r_real": [r_real],
    "pSA": [p_sa],
    "n_perm": [n_perm],
    "valid_null": [len(r_null_sa)]
}).to_csv(sum_path, index=False)

print(f"[CTRL1 DONE] pSA={p_sa:.6g}  valid_null={len(r_null_sa)}")
print("[SAVE]", null_path)
print("[SAVE]", sum_path)


==== [CTRL1] BrainSMASH SA-preserving null (small-ROI safe) ====
[INFO] ROI: 10 targetGenes: 1563 n_perm: 10000
[INFO] Note: ROI is small -> we will adapt knn/ns/nh per gene based on valid ROIs.
[CTRL1] 200/10000  valid=200
[CTRL1] 400/10000  valid=400
[CTRL1] 600/10000  valid=600
[CTRL1] 800/10000  valid=800
[CTRL1] 1000/10000  valid=1000
[CTRL1] 1200/10000  valid=1200
[CTRL1] 1400/10000  valid=1400
[CTRL1] 1600/10000  valid=1600
[CTRL1] 1800/10000  valid=1800
[CTRL1] 2000/10000  valid=2000
[CTRL1] 2200/10000  valid=2200
[CTRL1] 2400/10000  valid=2400
[CTRL1] 2600/10000  valid=2600
[CTRL1] 2800/10000  valid=2800
[CTRL1] 3000/10000  valid=3000
[CTRL1] 3200/10000  valid=3200
[CTRL1] 3400/10000  valid=3400
[CTRL1] 3600/10000  valid=3600
[CTRL1] 3800/10000  valid=3800
[CTRL1] 4000/10000  valid=4000
[CTRL1] 4200/10000  valid=4200
[CTRL1] 4400/10000  valid=4400
[CTRL1] 4600/10000  valid=4600
[CTRL1] 4800/10000  valid=4800
[CTRL1] 5000/10000  valid=5000
[CTRL1] 5200/10000  valid=5200
[CTRL1

In [13]:
print("r_real =", r_real)
print("null mean =", np.mean(r_null_sa), "null std =", np.std(r_null_sa))
print("null min/max =", np.min(r_null_sa), np.max(r_null_sa))
print("null 99.9% =", np.quantile(r_null_sa, 0.999))

r_real = 0.3549
null mean = 0.009590029389837537 null std = 0.0728590313788302
null min/max = -0.2448360239403693 0.2853743496556135
null 99.9% = 0.23055526580647442


In [14]:
z = (r_real - np.mean(r_null_sa)) / np.std(r_null_sa)
print("z =", z)

z = 4.73942576610338


# 4.Control-2 不同parcellation

In [12]:
import os
import glob
import inspect
import warnings
import numpy as np
import pandas as pd
import nibabel as nib
import abagen
from nilearn.maskers import NiftiLabelsMasker
from scipy import stats
from scipy.stats import pearsonr

# =========================================================
# 0) abagen与新版pandas兼容
# =========================================================
warnings.filterwarnings("ignore", message=".*groupby with axis=1 is deprecated.*", category=FutureWarning)

if "inplace" not in inspect.signature(pd.DataFrame.set_axis).parameters:
    _original_set_axis = pd.DataFrame.set_axis

    def _set_axis_compat(self, labels, axis=0, inplace=False, copy=None):
        result = _original_set_axis(self, labels, axis=axis, copy=copy)
        if inplace:
            if axis in (0, "index"): self.index = result.index
            else: self.columns = result.columns
            return None
        return result

    pd.DataFrame.set_axis = _set_axis_compat

if not hasattr(pd.DataFrame, "append"):
    def _dataframe_append_compat(self, other, ignore_index=False, verify_integrity=False, sort=False):
        others = list(other) if isinstance(other, (list, tuple)) else [other]
        return pd.concat([self] + others, ignore_index=ignore_index, verify_integrity=verify_integrity, sort=sort)

    pd.DataFrame.append = _dataframe_append_compat

if not hasattr(pd.Series, "append"):
    def _series_append_compat(self, to_append, ignore_index=False, verify_integrity=False):
        others = list(to_append) if isinstance(to_append, (list, tuple)) else [to_append]
        return pd.concat([self] + others, ignore_index=ignore_index, verify_integrity=verify_integrity)

    pd.Series.append = _series_append_compat

# =========================================================
# 1) 路径和参数
# =========================================================
base = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis"
ftract_probability_dir = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/f-tract/f-tract_v2112/ages_15_100/sr_8.40/seg_None_None/pl_200/Lausanne2008-125/export/probability"

atlas_path = os.path.join(base, "processed/01.HCP_FC/atl-Cammoun2012_space-MNI152NLin2009aSym_res-125_deterministic.nii")
atlas_info_path = os.path.join(base, "processed/01.HCP_FC/atl-Cammoun2012_space-MNI152NLin2009aSym_info.csv")
wm_map_path = os.path.join(base, "processed/06.active_map/res/working-memory/working-memory_association-test_z.nii.gz")
ahba_data_dir = os.path.join(base, "processed/07.AHBA_processing/data/modify_data/microarray")

gene_out_dir = os.path.join(base, "res/01.gene_selection")
coexp_out_dir = os.path.join(base, "res/02.co_expression")
corr_out_dir = os.path.join(base, "res/03.correlation")

for d in [gene_out_dir, coexp_out_dir, corr_out_dir]: os.makedirs(d, exist_ok=True)

expr125_path = os.path.join(gene_out_dir, "expr_Lausanne125_all_kwx.csv")
ccep125_path = os.path.join(gene_out_dir, "eeg_Lausanne125_all_kwx.csv")
n_perm = 10000
random_seed = 0
top_gene_fraction = 0.10
top_roi_fraction = 0.15

# =========================================================
# 2) 工具函数
# =========================================================
def make_ftract_alias(row):
    hemi, structure, label = str(row["hemisphere"]), str(row["structure"]), str(row["label"])
    if structure == "cortex": return ("lh." if hemi == "L" else "rh.") + label
    prefix = "Left-" if hemi == "L" else "Right-"
    sub_map = {"thalamusproper": "Thalamus-Proper", "caudate": "Caudate", "putamen": "Putamen", "pallidum": "Pallidum", "accumbensarea": "Accumbens-area", "hippocampus": "Hippocampus", "amygdala": "Amygdala", "brainstem": "Brain-Stem"}
    return sub_map[label] if label == "brainstem" else prefix + sub_map[label]

def coexp_from_zscored_expr(E):
    Z = stats.zscore(E.to_numpy(dtype=float), axis=0, nan_policy="omit", ddof=0)
    E_z = pd.DataFrame(Z, index=E.index, columns=E.columns)
    return E_z.T.corr(method="pearson")

def symmetrize_nan_matrix(M):
    X, XT = M.to_numpy(dtype=float), M.to_numpy(dtype=float).T
    count = np.isfinite(X).astype(int) + np.isfinite(XT).astype(int)
    total = np.where(np.isfinite(X), X, 0) + np.where(np.isfinite(XT), XT, 0)
    sym = np.full(X.shape, np.nan, dtype=float)
    np.divide(total, count, out=sym, where=count > 0)
    return pd.DataFrame(sym, index=M.index, columns=M.columns)

def corr_upper_triangle(A, B, min_edges=30):
    if A.shape != B.shape: raise ValueError(f"Shape mismatch: A{A.shape} vs B{B.shape}")
    if not A.index.equals(B.index) or not A.columns.equals(B.columns): raise ValueError("A和B的ROI顺序不一致。")
    iu = np.triu_indices(A.shape[0], k=1)
    a, b = A.to_numpy(dtype=float)[iu], B.to_numpy(dtype=float)[iu]
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < min_edges: return np.nan, np.nan, int(mask.sum())
    r, p = pearsonr(a[mask], b[mask])
    return float(r), float(p), int(mask.sum())

def qap_permutation_test(A, B, n_perm=10000, seed=0, min_edges=30):
    rng = np.random.default_rng(seed)
    X, Y = A.to_numpy(dtype=float), B.to_numpy(dtype=float)
    iu = np.triu_indices(X.shape[0], k=1)
    x_real, y_real = X[iu], Y[iu]
    mask_real = np.isfinite(x_real) & np.isfinite(y_real)

    if mask_real.sum() < min_edges: return np.nan, np.nan, np.nan, np.array([])

    r_real = pearsonr(x_real[mask_real], y_real[mask_real])[0]
    null_r = np.full(n_perm, np.nan)

    for p in range(n_perm):
        perm = rng.permutation(X.shape[0])
        x_perm = X[np.ix_(perm, perm)][iu]
        mask = np.isfinite(x_perm) & np.isfinite(y_real)

        if mask.sum() >= min_edges and np.nanstd(x_perm[mask]) > 0 and np.nanstd(y_real[mask]) > 0:
            null_r[p] = pearsonr(x_perm[mask], y_real[mask])[0]

        if (p + 1) % 2000 == 0: print(f"[PERM] {p + 1}/{n_perm}")

    null_r = null_r[np.isfinite(null_r)]
    p_two = (np.sum(np.abs(null_r) >= abs(r_real)) + 1) / (len(null_r) + 1)
    p_greater = (np.sum(null_r >= r_real) + 1) / (len(null_r) + 1)

    return float(r_real), float(p_two), float(p_greater), null_r

# =========================================================
# 3) Lausanne125标签
# =========================================================
print("==== [INFO] Lausanne125 atlas ====")

atlas_info_all = pd.read_csv(atlas_info_path)
atlas_info = atlas_info_all.loc[atlas_info_all["scale"].eq("scale125")].copy()
atlas_info["id"] = pd.to_numeric(atlas_info["id"]).astype(int)
atlas_info["alias"] = atlas_info.apply(make_ftract_alias, axis=1)
atlas_info = atlas_info.sort_values("id")
id_to_alias = dict(zip(atlas_info["id"], atlas_info["alias"]))

print("[INFO] atlas ROIs:", len(atlas_info))

# =========================================================
# 4) 重建或读取Lausanne125 CCEP
# =========================================================
print("\n==== [CCEP] Lausanne125 probability matrix ====")

if os.path.exists(ccep125_path):
    CCEP = pd.read_csv(ccep125_path, index_col=0).apply(pd.to_numeric, errors="coerce")
    CCEP.index = CCEP.index.astype(str).str.strip()
    CCEP.columns = CCEP.columns.astype(str).str.strip()
    print("[LOAD]", CCEP.shape)
else:
    files = sorted(glob.glob(os.path.join(ftract_probability_dir, "*", "*__probability__efferent.txt")))
    files = [f for f in files if not os.path.basename(f).startswith("._")]

    if len(files) == 0: raise FileNotFoundError(f"没有找到F-TRACT文件：{ftract_probability_dir}")

    rows = {}

    for i, f in enumerate(files):
        roi = os.path.basename(os.path.dirname(f))
        values = pd.read_csv(f, sep="\t", header=None, index_col=0).iloc[:, 0]
        values.index = values.index.astype(str).str.strip()
        rows[roi] = pd.to_numeric(values, errors="coerce")

        if (i + 1) % 50 == 0: print(f"[CCEP] {i + 1}/{len(files)}")

    CCEP = pd.DataFrame.from_dict(rows, orient="index")
    CCEP.index = CCEP.index.astype(str).str.strip()
    CCEP.columns = CCEP.columns.astype(str).str.strip()
    common = CCEP.index.intersection(CCEP.columns)
    CCEP = CCEP.loc[common, common]
    CCEP.to_csv(ccep125_path)

    print("[SAVE]", ccep125_path)

# =========================================================
# 5) 生成或读取Lausanne125 AHBA表达
# =========================================================
print("\n==== [AHBA] Lausanne125 expression ====")

if os.path.exists(expr125_path):
    expression = pd.read_csv(expr125_path, index_col=0).apply(pd.to_numeric, errors="coerce")
    expression.index = expression.index.astype(str).str.strip()
    print("[LOAD]", expression.shape)
else:
    atlas_info_abagen = atlas_info[["id", "label", "hemisphere", "structure"]].copy()
    expression = abagen.get_expression_data(atlas_path, atlas_info=atlas_info_abagen, donors=["9861", "10021", "12876", "14380", "15496", "15697"], data_dir=ahba_data_dir, missing="centroids")
    expression.index = pd.to_numeric(expression.index, errors="raise").astype(int)
    expression.index = expression.index.map(id_to_alias)
    expression.index.name = "ROI"
    expression.to_csv(expr125_path)

    print("[SAVE]", expr125_path)

expression.columns = expression.columns.astype(str).str.strip()

# =========================================================
# 6) 工作记忆图投影到Lausanne125
# =========================================================
print("\n==== [WM] Lausanne125 working-memory scores ====")

atlas_img = nib.load(atlas_path)
wm_img = nib.load(wm_map_path)
masker = NiftiLabelsMasker(labels_img=atlas_img, labels=atlas_info["alias"].tolist(), resampling_target="data", strategy="mean")
wm_scores = masker.fit_transform(wm_img).ravel()

if len(wm_scores) != len(atlas_info): raise ValueError(f"WM ROI数量错误：{len(wm_scores)} vs {len(atlas_info)}")

wm_roi = atlas_info.copy()
wm_roi["wm_z_mean"] = wm_scores
wm_roi.to_csv(os.path.join(gene_out_dir, "Lausanne125_working_memory_ROI_scores_kwx.csv"), index=False)

left_info = wm_roi.loc[(wm_roi["hemisphere"] == "L") & (wm_roi["structure"] == "cortex")].copy()
left_info = left_info.dropna(subset=["wm_z_mean"]).sort_values("wm_z_mean", ascending=False)
left_rois = left_info["alias"].tolist()

print("[INFO] left cortical ROIs:", len(left_rois))

# =========================================================
# 7) 在Lausanne125重新计算WM–基因相关并选前10%
# =========================================================
print("\n==== [GENE] Recalculate Lausanne125 WM-related genes ====")

gene_selection_rois = pd.Index(left_rois).intersection(expression.index)
expr_left = expression.loc[gene_selection_rois]
wm_left = left_info.set_index("alias").loc[gene_selection_rois, "wm_z_mean"]

pearson_gene_r = expr_left.corrwith(wm_left, axis=0, method="pearson")
spearman_gene_r = expr_left.corrwith(wm_left, axis=0, method="spearman")

gene_scores = pd.DataFrame({"pearson_r": pearson_gene_r, "spearman_r": spearman_gene_r})
gene_scores["abs_pearson_r"] = gene_scores["pearson_r"].abs()
gene_scores["abs_spearman_r"] = gene_scores["spearman_r"].abs()
gene_scores = gene_scores.sort_values("abs_pearson_r", ascending=False)

n_top_genes = int(expression.shape[1] * top_gene_fraction)
top_genes = gene_scores.dropna(subset=["abs_pearson_r"]).head(n_top_genes).index.tolist()

if len(top_genes) < 2: raise ValueError("筛选到的WM相关基因太少。")

gene_scores.to_csv(os.path.join(gene_out_dir, "gene_scores_Lausanne125_WM_kwx.csv"))
expr_left_top10 = expr_left.loc[:, top_genes]
expr_left_top10.to_csv(os.path.join(gene_out_dir, "expr_Lausanne125_left_top10_WM_genes_kwx.csv"))

print("[INFO] all genes:", expression.shape[1])
print("[INFO] selected top10% genes:", len(top_genes))
print("[INFO] strongest genes:", top_genes[:10])

# =========================================================
# 8) 选择WM值最高的前15%脑区
# =========================================================
n_top_rois = int(np.ceil(len(left_info) * top_roi_fraction))
top15_info = left_info.head(n_top_rois).copy()
top15_rois = top15_info["alias"].tolist()

top15_info.to_csv(os.path.join(gene_out_dir, "Lausanne125_left_top15_working_memory_ROIs_kwx.csv"), index=False)
expr_left_top10.loc[top15_rois].to_csv(os.path.join(gene_out_dir, "expr_Lausanne125_left_top10_WM_top15ROIs_kwx.csv"))

print("[INFO] WM top15% ROIs:", len(top15_rois))
print(top15_info[["id", "alias", "wm_z_mean"]].to_string(index=False))

# =========================================================
# 9) 计算共表达、CCEP相关和P值
# =========================================================
print("\n==== [CTRL2] Full Lausanne125 pipeline robustness ====")

analysis_versions = {"Lausanne125_L_all_cortex_rederived_genes": left_rois, "Lausanne125_L_WM_top15pct_rederived_genes": top15_rois}
ctrl2_rows = []

for version_index, (name, requested_rois) in enumerate(analysis_versions.items()):
    common_rois = pd.Index(requested_rois).intersection(expr_left_top10.index).intersection(CCEP.index).intersection(CCEP.columns)

    if len(common_rois) < 9:
        print(f"[SKIP] {name}: common ROI={len(common_rois)}")
        continue

    Et = expr_left_top10.loc[common_rois, top_genes]
    Fc_directed = CCEP.loc[common_rois, common_rois]
    Fc = symmetrize_nan_matrix(Fc_directed)
    coexp = coexp_from_zscored_expr(Et)

    r_v, p_pearson, n_edges = corr_upper_triangle(coexp, Fc, min_edges=30)

    print(f"\n[CTRL2] {name}: ROI={len(common_rois)} genes={len(top_genes)} r={r_v:.6f} Pearson_P={p_pearson:.6g} edges={n_edges}")
    print(f"[CTRL2] Starting {n_perm} QAP permutations...")

    r_perm, p_perm_two, p_perm_greater, null_r = qap_permutation_test(coexp, Fc, n_perm=n_perm, seed=random_seed + version_index, min_edges=30)

    safe_name = name.replace(" ", "_")
    Et.to_csv(os.path.join(gene_out_dir, f"expr_{safe_name}_kwx.csv"))
    coexp.to_csv(os.path.join(coexp_out_dir, f"gene_coexpr_{safe_name}_kwx.csv"))
    Fc.to_csv(os.path.join(corr_out_dir, f"eeg_{safe_name}_symmetrized_kwx.csv"))
    pd.Series(null_r, name="r_null").to_csv(os.path.join(corr_out_dir, f"ctrl2_null_{safe_name}_kwx.csv"), index=False)

    ctrl2_rows.append((name, len(common_rois), len(top_genes), r_v, p_pearson, p_perm_two, p_perm_greater, n_perm, len(null_r), n_edges))
    print(f"[DONE] r={r_v:.6f} p_perm_two={p_perm_two:.6g} p_perm_greater={p_perm_greater:.6g}")

# =========================================================
# 10) 保存汇总
# =========================================================
ctrl2_summary = pd.DataFrame(ctrl2_rows, columns=["version", "n_roi", "n_genes", "r", "p_pearson", "p_perm_two_sided", "p_perm_greater", "n_perm", "valid_perm", "n_edges"])
ctrl2_summary.to_csv(os.path.join(corr_out_dir, "ctrl2_parcellation_robustness_kwx.csv"), index=False)

print("\n[SAVE] ctrl2_parcellation_robustness_kwx.csv")
print(ctrl2_summary.to_string(index=False))
print("\n==== CTRL2 FINISHED ====")

==== [INFO] Lausanne125 atlas ====
[INFO] atlas ROIs: 234

==== [CCEP] Lausanne125 probability matrix ====
[CCEP] 50/235
[CCEP] 100/235
[CCEP] 150/235
[CCEP] 200/235
[SAVE] /Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/eeg_Lausanne125_all_kwx.csv

==== [AHBA] Lausanne125 expression ====


/opt/anaconda3/lib/python3.13/site-packages/abagen/samples_.py:405: FutureWarning: The provided callable <function mean at 0x122641760> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .aggregate(metric)
/var/folders/42/hpvfc_yd075_rrzsh1__k5_c0000gn/T/ipykernel_85848/620105813.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([self] + others, ignore_index=ignore_index, verify_integrity=verify_integrity, sort=sort)
/opt/anaconda3/lib/python3.13/site-packages/abagen/samples_.py:405: FutureWarning: The provided callable <function mean at 0x122641760> is currently using DataFrameGroupBy.mean. In a 

[SAVE] /Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_Lausanne125_all_kwx.csv

==== [WM] Lausanne125 working-memory scores ====
[INFO] left cortical ROIs: 111

==== [GENE] Recalculate Lausanne125 WM-related genes ====
[INFO] all genes: 15633
[INFO] selected top10% genes: 1563
[INFO] strongest genes: ['STARD4', 'DYNLT3', 'CACNG6', 'LYNX1', 'MED31', 'MYCN', 'PHLDA2', 'NFIB', 'GPR20', 'DENND2D']
[INFO] WM top15% ROIs: 17
 id                     alias  wm_z_mean
174     lh.superiorparietal_3   4.351355
127 lh.rostralmiddlefrontal_1   3.998647
144  lh.caudalmiddlefrontal_3   3.395176
143  lh.caudalmiddlefrontal_2   2.908395
151           lh.precentral_7   2.504736
171        lh.supramarginal_5   2.481350
126      lh.parsopercularis_2   2.407539
177     lh.superiorparietal_6   2.347299
129 lh.rostralmiddlefrontal_3   2.146200
226               lh.insula_4   2.125187
149           lh.precentral_5   2.054530
176     lh.superiorparietal_5   1.9

# 5.Control-3 随机选择基因

In [6]:
# =========================================================
# 7) 控制实验 3：从全部基因中完全随机抽取，重复 10,000 次
# =========================================================
print("\n==== [CTRL3] Completely random genes null ====")

n_perm = 10000
rng = np.random.default_rng(0)
Gt = E_target.shape[1]
all_genes = np.asarray(E_all.columns)

print("[INFO] number of target genes:", Gt)
print("[INFO] total random gene pool:", len(all_genes))

if Gt > len(all_genes):
    raise ValueError(f"目标基因数量 ({Gt}) 大于全基因池数量 ({len(all_genes)})，无法无放回抽样。")

r_null_rand = np.full(n_perm, np.nan, dtype=float)

for p in range(n_perm):
    sel = rng.choice(all_genes, size=Gt, replace=False)
    E_rand = E_all.loc[:, sel]
    coexp_rand = coexp_from_expr(E_rand, method="pearson")
    r_p, n_edges = corr_upper_triangle(coexp_rand, FC, min_edges=30)
    r_null_rand[p] = r_p

    if (p + 1) % 1000 == 0:
        print(f"[CTRL3] {p + 1}/{n_perm}")

r_null_rand = r_null_rand[np.isfinite(r_null_rand)]

if len(r_null_rand) == 0:
    raise ValueError("所有随机结果均为 NaN，请检查表达矩阵和 FC 矩阵。")

p_rand = np.mean(r_null_rand >= r_real)

pd.Series(r_null_rand, name="r_null_random_genes").to_csv(os.path.join(out_dir, "ctrl3_null_random_genes_kwx.csv"), index=False)
pd.DataFrame({"r_real": [r_real], "p_random": [p_rand], "n_perm": [n_perm], "valid_null": [len(r_null_rand)]}).to_csv(os.path.join(out_dir, "ctrl3_summary_random_genes_kwx.csv"), index=False)

print(f"[CTRL3 DONE] p_random={p_rand:.6g}  valid_null={len(r_null_rand)}")
print("[SAVE] ctrl3_null_random_genes_kwx.csv, ctrl3_summary_random_genes_kwx.csv")


==== [CTRL3] Completely random genes null ====
[INFO] number of target genes: 1563
[INFO] total random gene pool: 15634
[CTRL3] 1000/10000
[CTRL3] 2000/10000
[CTRL3] 3000/10000
[CTRL3] 4000/10000
[CTRL3] 5000/10000
[CTRL3] 6000/10000
[CTRL3] 7000/10000
[CTRL3] 8000/10000
[CTRL3] 9000/10000
[CTRL3] 10000/10000
[CTRL3 DONE] p_random=0  valid_null=10000
[SAVE] ctrl3_null_random_genes_kwx.csv, ctrl3_summary_random_genes_kwx.csv


# 6.Control-4 阈值非目标基因

In [7]:
# =========================================================
# 8) 控制实验 4：阈值随机基因 null
# =========================================================
print("\n==== [CTRL4] Thresholded random genes null ====")

thresholds = [0.0, 0.5, 1.0]
n_perm = 10000
rng = np.random.default_rng(0)
Gt = E_target.shape[1]
all_genes = list(E_all.columns)

# 计算每个基因在所有 ROI 上的平均表达
gene_mean = E_all.mean(axis=0, skipna=True)

ctrl4_rows = []

for t in thresholds:
    # 从全部基因中筛选平均表达达到阈值的基因，包括目标基因
    pool_t = [g for g in all_genes if gene_mean.get(g, -np.inf) >= t]
    print(f"[CTRL4] threshold={t}  pool_t={len(pool_t)}")

    if len(pool_t) < Gt:
        print(f"[CTRL4] threshold={t} pool too small (< targetGenes). Skip.")
        continue

    r_null_t = np.full(n_perm, np.nan, dtype=float)

    for p in range(n_perm):
        sel = rng.choice(pool_t, size=Gt, replace=False)
        E_rand = E_all.loc[:, sel]
        coexp_rand = coexp_from_expr(E_rand, method="pearson")
        r_p, n_edges = corr_upper_triangle(coexp_rand, FC, min_edges=30)
        r_null_t[p] = r_p

        if (p + 1) % 2000 == 0:
            print(f"[CTRL4 t={t}] {p + 1}/{n_perm}")

    r_null_t = r_null_t[np.isfinite(r_null_t)]

    if len(r_null_t) == 0:
        print(f"[CTRL4] threshold={t} produced no valid null results. Skip.")
        continue

    p_t = np.mean(r_null_t >= r_real)

    fn = f"ctrl4_null_threshold_{str(t).replace('.', 'p')}.csv"
    pd.Series(r_null_t, name=f"r_null_thr_{t}").to_csv(os.path.join(out_dir, fn), index=False)

    ctrl4_rows.append((t, len(pool_t), len(r_null_t), p_t))
    print(f"[CTRL4 DONE] threshold={t}  p_thr={p_t:.6g}  valid_null={len(r_null_t)}  saved={fn}")

pd.DataFrame(ctrl4_rows, columns=["threshold", "pool_size", "valid_null", "p_threshold"]).to_csv(os.path.join(out_dir, "ctrl4_summary_thresholds_kwx.csv"), index=False)

print("[SAVE] ctrl4_summary_thresholds_kwx.csv (+ each threshold null csv)")
print("\n==== ALL CONTROLS FINISHED ====")


==== [CTRL4] Thresholded random genes null ====
[CTRL4] threshold=0.0  pool_t=15634
[CTRL4 t=0.0] 2000/10000
[CTRL4 t=0.0] 4000/10000
[CTRL4 t=0.0] 6000/10000
[CTRL4 t=0.0] 8000/10000
[CTRL4 t=0.0] 10000/10000
[CTRL4 DONE] threshold=0.0  p_thr=0  valid_null=10000  saved=ctrl4_null_threshold_0p0.csv
[CTRL4] threshold=0.5  pool_t=7427
[CTRL4 t=0.5] 2000/10000
[CTRL4 t=0.5] 4000/10000
[CTRL4 t=0.5] 6000/10000
[CTRL4 t=0.5] 8000/10000
[CTRL4 t=0.5] 10000/10000
[CTRL4 DONE] threshold=0.5  p_thr=0.0001  valid_null=10000  saved=ctrl4_null_threshold_0p5.csv
[CTRL4] threshold=1.0  pool_t=0
[CTRL4] threshold=1.0 pool too small (< targetGenes). Skip.
[SAVE] ctrl4_summary_thresholds_kwx.csv (+ each threshold null csv)

==== ALL CONTROLS FINISHED ====


# 7.Control-5 随机抽取10个脑区

In [2]:
import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, zscore

# =========================================================
# 控制实验 5：从非 WM 的 54 个左脑 ROI 中随机抽 10 ROI
# 每次重新构建基因共表达矩阵，并与对应 EEG 子矩阵相关
# =========================================================
print("\n==== [CTRL5] Random 10 non-WM ROI from left 64 ROI ====")

# 64个左脑ROI的目标基因表达矩阵：64 ROI × targetGenes
E_left64_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_left_top10_activate_kwx.csv"

# 64个左脑ROI的EEG矩阵：64 ROI × 64 ROI
FC_left64_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/eeg_left_kwx.csv"

# 真实10个WM ROI表达矩阵：10 ROI × targetGenes，用它的index定义需要排除的WM ROI
E_wm10_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_left_top10_top15_activate_kwx.csv"

# 真实主结果相关
r_real = 0.3549

# =========================================================
# 工具函数
# =========================================================
def corr_upper_triangle(A: pd.DataFrame, B: pd.DataFrame, min_edges: int = 30):
    if A.shape != B.shape: raise ValueError(f"Shape mismatch: A{A.shape} vs B{B.shape}")
    R = A.shape[0]; iu = np.triu_indices(R, k=1); a = A.values[iu]; b = B.values[iu]; mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < min_edges: return np.nan, int(mask.sum())
    return pearsonr(a[mask], b[mask])[0], int(mask.sum())

def coexp_from_expr_zscore(E: pd.DataFrame, method: str = "pearson"):
    E_z = pd.DataFrame(zscore(E, axis=0, nan_policy="omit"), index=E.index, columns=E.columns)
    E_z = E_z.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="any")
    return E_z.T.corr(method=method)

# =========================================================
# 读取数据
# =========================================================
E64 = pd.read_csv(E_left64_path, index_col=0).apply(pd.to_numeric, errors="coerce")
FC64 = pd.read_csv(FC_left64_path, index_col=0).apply(pd.to_numeric, errors="coerce")
E_wm10 = pd.read_csv(E_wm10_path, index_col=0).apply(pd.to_numeric, errors="coerce")

E64.index = E64.index.astype(str).str.strip()
FC64.index = FC64.index.astype(str).str.strip()
FC64.columns = FC64.columns.astype(str).str.strip()
E_wm10.index = E_wm10.index.astype(str).str.strip()

# =========================================================
# 对齐64个左脑ROI
# =========================================================
comm64 = E64.index.intersection(FC64.index).intersection(FC64.columns)
E64 = E64.loc[comm64, :]
FC64 = FC64.loc[comm64, comm64]

# =========================================================
# 排除真实10个WM ROI，得到非WM ROI随机池
# =========================================================
wm_rois = E_wm10.index.intersection(comm64)
non_wm_rois = pd.Index([r for r in comm64 if r not in set(wm_rois)])

print("[INFO] available left ROIs:", len(comm64))
print("[INFO] WM ROIs excluded:", len(wm_rois))
print("[INFO] non-WM ROI pool:", len(non_wm_rois))

print("\n[INFO] excluded WM ROIs:")
print(list(wm_rois))

if len(wm_rois) != 10: print("[WARNING] WM ROI数量不是10个，请检查 E_wm10_path 是否是你的真实10个WM ROI文件。")
if len(non_wm_rois) < 10: raise ValueError("非WM ROI少于10个，无法随机抽样。")

# =========================================================
# 随机抽样：每次从54个非WM ROI中抽10个ROI
# =========================================================
n_iter = 10000
n_pick = 10
seed = 20260706
rng = np.random.default_rng(seed)
roi_pool = np.array(non_wm_rois)

random_rows = []

for i in range(n_iter):
    picked_rois = rng.choice(roi_pool, size=n_pick, replace=False)
    Et_rand = E64.loc[picked_rois, :]
    Fc_rand = FC64.loc[picked_rois, picked_rois]

    # 关键：每次随机抽取10个非WM ROI后，重新z-score并构建这10个ROI的基因共表达矩阵
    coexp_rand = coexp_from_expr_zscore(Et_rand, method="pearson")

    # 将随机ROI的基因共表达矩阵与对应的EEG子矩阵做相关
    r_rand, n_edges = corr_upper_triangle(coexp_rand, Fc_rand, min_edges=30)

    random_rows.append({"iter": i + 1, "n_roi": n_pick, "n_genes": Et_rand.shape[1], "n_genes_after_zscore": coexp_rand.shape[0], "r": r_rand, "n_edges": n_edges, "rois": ";".join(picked_rois)})

    if (i + 1) % 1000 == 0: print(f"[CTRL5] {i+1}/{n_iter}")

ctrl5_random_nonwm_df = pd.DataFrame(random_rows)
valid_r = ctrl5_random_nonwm_df["r"].dropna().values

# =========================================================
# 输出结果
# =========================================================
print("\n==== [CTRL5 SUMMARY] ====")
print("[CTRL5] valid random samples:", len(valid_r), "/", n_iter)
print("[CTRL5] random r mean:", np.nanmean(valid_r))
print("[CTRL5] random r std :", np.nanstd(valid_r))
print("[CTRL5] random r 2.5%, 50%, 97.5%:", np.nanpercentile(valid_r, [2.5, 50, 97.5]))

p_emp_right = (np.sum(valid_r >= r_real) + 1) / (len(valid_r) + 1)
p_emp_two_sided = (np.sum(np.abs(valid_r) >= abs(r_real)) + 1) / (len(valid_r) + 1)

print("[CTRL5] r_real:", r_real)
print("[CTRL5] empirical p, right-tailed:", p_emp_right)
print("[CTRL5] empirical p, two-sided   :", p_emp_two_sided)

print("\n[CTRL5] first 5 random samples:")
print(ctrl5_random_nonwm_df.head())


==== [CTRL5] Random 10 non-WM ROI from left 64 ROI ====
[INFO] available left ROIs: 64
[INFO] WM ROIs excluded: 10
[INFO] non-WM ROI pool: 54

[INFO] excluded WM ROIs:
['lh.parsopercularis_1', 'lh.rostralmiddlefrontal_1', 'lh.rostralmiddlefrontal_2', 'lh.superiorfrontal_3', 'lh.superiorfrontal_4', 'lh.caudalmiddlefrontal_1', 'lh.precentral_3', 'lh.precentral_4', 'lh.superiorparietal_2', 'lh.superiorparietal_3']
[CTRL5] 1000/10000
[CTRL5] 2000/10000
[CTRL5] 3000/10000
[CTRL5] 4000/10000
[CTRL5] 5000/10000
[CTRL5] 6000/10000
[CTRL5] 7000/10000
[CTRL5] 8000/10000
[CTRL5] 9000/10000
[CTRL5] 10000/10000

==== [CTRL5 SUMMARY] ====
[CTRL5] valid random samples: 5283 / 10000
[CTRL5] random r mean: 0.23766611972246499
[CTRL5] random r std : 0.17966042003276375
[CTRL5] random r 2.5%, 50%, 97.5%: [-0.12867697  0.24531217  0.56505637]
[CTRL5] r_real: 0.3549
[CTRL5] empirical p, right-tailed: 0.27403482210446634
[CTRL5] empirical p, two-sided   : 0.2746025738077214

[CTRL5] first 5 random samples:

# 8.Control-6 不同选取范围脑区

In [13]:
import os
import numpy as np
import pandas as pd
from scipy.stats import zscore, pearsonr

# =========================================================
# 0) 路径
# =========================================================
base = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis"

expr_path = os.path.join(base, "res/01.gene_selection/expr_left_top10_activate_kwx.csv")
eeg_path = os.path.join(base, "res/01.gene_selection/eeg_left_kwx.csv")
wm_path = os.path.join(base, "processed/06.active_map/res/roi_map/working-memory_roi.zscore_kwx.csv")

out_dir = os.path.join(base, "res/03.correlation")
os.makedirs(out_dir, exist_ok=True)

summary_path = os.path.join(out_dir, "ctrl6_Lausanne60_roi_proportion_curve_kwx.csv")
membership_path = os.path.join(out_dir, "ctrl6_Lausanne60_roi_proportion_membership_kwx.csv")

# =========================================================
# 1) 脑区比例：累计前5%至100%
# =========================================================
percentages = np.arange(5, 101, 5)

# =========================================================
# 2) 读取数据
# =========================================================
expr = pd.read_csv(expr_path, index_col=0).apply(pd.to_numeric, errors="coerce")
expr.index = expr.index.astype(str).str.strip()

eeg = pd.read_csv(eeg_path, index_col=0).apply(pd.to_numeric, errors="coerce")
eeg.index = eeg.index.astype(str).str.strip()
eeg.columns = eeg.columns.astype(str).str.strip()

# 原始文件前64区为右脑、64:128为左脑
wm = pd.read_csv(wm_path, index_col=0, header=None).iloc[64:128, 0]
wm.index = wm.index.astype(str).str.strip()
wm = pd.to_numeric(wm, errors="coerce").dropna().sort_values(ascending=False)

# 确保三种数据拥有相同的64个左脑ROI
available_rois = [roi for roi in wm.index if roi in expr.index and roi in eeg.index and roi in eeg.columns]
wm = wm.loc[available_rois]

print("[INFO] Lausanne60 left ROIs:", len(available_rois))
print("[INFO] fixed target genes:", expr.shape[1])

if len(available_rois) != 64:
    print(f"[WARNING] 预期64个ROI，实际匹配到{len(available_rois)}个。")

# =========================================================
# 3) 不同比例下计算基因共表达–EEG相关
# =========================================================
summary_rows = []
membership_rows = []

for percent in percentages:
    n_roi = int(np.ceil(len(available_rois) * percent / 100))
    selected_set = set(wm.index[:n_roi])

    # 保持原始表达矩阵顺序，与原top15代码一致
    selected_rois = [roi for roi in expr.index if roi in selected_set and roi in eeg.index and roi in eeg.columns]

    expr_selected = expr.loc[selected_rois]
    eeg_selected = eeg.loc[selected_rois, selected_rois]

    # 对每个基因在当前ROI集合中重新做z-score
    expr_z = pd.DataFrame(zscore(expr_selected, axis=0, nan_policy="omit", ddof=0), index=expr_selected.index, columns=expr_selected.columns)

    # ROI × ROI基因共表达矩阵
    gene_coexpression = expr_z.T.corr(method="pearson")

    # 严格复现原始方法：使用未对称化EEG矩阵的上三角
    iu = np.triu_indices(len(selected_rois), k=1)
    gene_edges = gene_coexpression.to_numpy(dtype=float)[iu]
    eeg_edges = eeg_selected.to_numpy(dtype=float)[iu]
    mask = np.isfinite(gene_edges) & np.isfinite(eeg_edges)
    n_edges = int(mask.sum())

    if n_edges >= 3 and np.std(gene_edges[mask]) > 0 and np.std(eeg_edges[mask]) > 0:
        r_value, p_pearson = pearsonr(gene_edges[mask], eeg_edges[mask])
    else:
        r_value, p_pearson = np.nan, np.nan

    summary_rows.append((percent / 100, percent, n_roi, len(selected_rois), expr_selected.shape[1], r_value, p_pearson, n_edges))

    for rank, roi in enumerate(wm.index[:n_roi], start=1):
        membership_rows.append((percent, rank, roi, wm.loc[roi]))

    print(f"[ROI {percent:3d}%] requested={n_roi:2d} matched={len(selected_rois):2d} edges={n_edges:4d} r={r_value:.6f} p={p_pearson:.6g}")

# =========================================================
# 4) 保存
# =========================================================
summary = pd.DataFrame(summary_rows, columns=["proportion", "percent", "n_roi_requested", "n_roi_used", "n_genes", "r", "p_pearson", "n_edges"])
membership = pd.DataFrame(membership_rows, columns=["percent", "roi_rank", "roi", "wm_z_mean"])

summary.to_csv(summary_path, index=False)
membership.to_csv(membership_path, index=False)

print("\n[SAVE]", summary_path)
print("[SAVE]", membership_path)
print(summary.to_string(index=False))

# 验证15%是否复现原结果
r_15 = summary.loc[summary["percent"] == 15, "r"].iloc[0]
print(f"\n[CHECK] Top15% r={r_15:.6f}; expected approximately 0.3549")

[INFO] Lausanne60 left ROIs: 64
[INFO] fixed target genes: 1563
[ROI   5%] requested= 4 matched= 4 edges=   6 r=0.128901 p=0.807719
[ROI  10%] requested= 7 matched= 7 edges=  19 r=0.312301 p=0.193012
[ROI  15%] requested=10 matched=10 edges=  43 r=0.354870 p=0.0195414
[ROI  20%] requested=13 matched=13 edges=  74 r=0.276380 p=0.0171412
[ROI  25%] requested=16 matched=16 edges= 104 r=0.182550 p=0.0636297
[ROI  30%] requested=20 matched=20 edges= 145 r=0.155213 p=0.0623016
[ROI  35%] requested=23 matched=23 edges= 195 r=0.213939 p=0.00267216
[ROI  40%] requested=26 matched=26 edges= 231 r=0.226929 p=0.000509403
[ROI  45%] requested=29 matched=29 edges= 305 r=0.281224 p=5.96595e-07
[ROI  50%] requested=32 matched=32 edges= 379 r=0.325328 p=8.58757e-11
[ROI  55%] requested=36 matched=36 edges= 481 r=0.309875 p=3.64751e-12
[ROI  60%] requested=39 matched=39 edges= 553 r=0.304072 p=2.70915e-13
[ROI  65%] requested=42 matched=42 edges= 651 r=0.305383 p=1.61409e-15
[ROI  70%] requested=45 matc

# 9.Control-7 不同z值选脑区计算相关

In [14]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img
from scipy.stats import zscore, pearsonr

# =========================================================
# 0) 路径和参数
# =========================================================
base = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis"

atlas_path = os.path.join(base, "rawdata/MNI152NLin2009aSym_lausenne60.nii.gz")
atlas_info_path = os.path.join(base, "rawdata/lausanne_scale60_atlas_info.csv")
wm_map_path = os.path.join(base, "processed/06.active_map/res/working-memory/working-memory_association-test_z.nii.gz")
expr_path = os.path.join(base, "res/01.gene_selection/expr_left_top10_activate_kwx.csv")
eeg_path = os.path.join(base, "res/01.gene_selection/eeg_left_kwx.csv")

z_thresholds = [2, 3, 4, 5]
n_perm = 10000
random_seed = 0
min_edges = 30

# =========================================================
# 1) 工具函数
# =========================================================
def calculate_coexpression(E):
    Z = zscore(E.to_numpy(dtype=float), axis=0, nan_policy="omit", ddof=0)
    E_z = pd.DataFrame(Z, index=E.index, columns=E.columns)
    return E_z.T.corr(method="pearson")

def corr_upper_triangle(A, B, min_edges=30):
    iu = np.triu_indices(A.shape[0], k=1)
    a, b = A.to_numpy(dtype=float)[iu], B.to_numpy(dtype=float)[iu]
    mask = np.isfinite(a) & np.isfinite(b)

    if mask.sum() < min_edges or np.std(a[mask]) == 0 or np.std(b[mask]) == 0:
        return np.nan, np.nan, int(mask.sum())

    r, p = pearsonr(a[mask], b[mask])
    return float(r), float(p), int(mask.sum())

def qap_test(A, B, n_perm=10000, seed=0, min_edges=30):
    rng = np.random.default_rng(seed)
    X, Y = A.to_numpy(dtype=float), B.to_numpy(dtype=float)
    iu = np.triu_indices(X.shape[0], k=1)
    x_real, y_real = X[iu], Y[iu]
    mask_real = np.isfinite(x_real) & np.isfinite(y_real)

    if mask_real.sum() < min_edges:
        return np.nan, np.nan, np.nan, 0

    r_real = pearsonr(x_real[mask_real], y_real[mask_real])[0]
    null_r = np.full(n_perm, np.nan)

    for p in range(n_perm):
        perm = rng.permutation(X.shape[0])
        x_perm = X[np.ix_(perm, perm)][iu]
        mask = np.isfinite(x_perm) & np.isfinite(y_real)

        if mask.sum() >= min_edges and np.std(x_perm[mask]) > 0 and np.std(y_real[mask]) > 0:
            null_r[p] = pearsonr(x_perm[mask], y_real[mask])[0]

    null_r = null_r[np.isfinite(null_r)]

    if len(null_r) == 0:
        return float(r_real), np.nan, np.nan, 0

    p_two = (np.sum(np.abs(null_r) >= abs(r_real)) + 1) / (len(null_r) + 1)
    p_greater = (np.sum(null_r >= r_real) + 1) / (len(null_r) + 1)

    return float(r_real), float(p_two), float(p_greater), len(null_r)

# =========================================================
# 2) 读取数据
# =========================================================
atlas_img = nib.load(atlas_path)
wm_img = nib.load(wm_map_path)
atlas_info = pd.read_csv(atlas_info_path)

# 与原分析一致：64:128为左脑64区
left_info = atlas_info.iloc[64:128].copy()
left_info["id"] = pd.to_numeric(left_info["id"]).astype(int)
left_info["alias1"] = left_info["alias1"].astype(str).str.strip()

# 将Lausanne60离散标签重采样到WM图空间
atlas_resampled = resample_to_img(atlas_img, wm_img, interpolation="nearest", force_resample=True, copy_header=True)
atlas_data = np.rint(np.asanyarray(atlas_resampled.dataobj)).astype(int)
wm_data = np.asanyarray(wm_img.dataobj).astype(float)

expr = pd.read_csv(expr_path, index_col=0).apply(pd.to_numeric, errors="coerce")
expr.index = expr.index.astype(str).str.strip()

eeg = pd.read_csv(eeg_path, index_col=0).apply(pd.to_numeric, errors="coerce")
eeg.index = eeg.index.astype(str).str.strip()
eeg.columns = eeg.columns.astype(str).str.strip()

print("Expression shape:", expr.shape)
print("EEG shape:", eeg.shape)
print("WM voxel Z range:", np.nanmin(wm_data), "to", np.nanmax(wm_data))

# =========================================================
# 3) 不同Z阈值分析
# =========================================================
summary_rows = []

for threshold_index, threshold in enumerate(z_thresholds):
    roi_rows = []

    for _, row in left_info.iterrows():
        roi_id = int(row["id"])
        roi_name = row["alias1"]
        roi_mask = atlas_data == roi_id
        n_voxels = int(roi_mask.sum())
        n_above = int(np.sum(roi_mask & (wm_data >= threshold)))
        fraction_above = n_above / n_voxels if n_voxels > 0 else np.nan
        mean_z = np.nanmean(wm_data[roi_mask]) if n_voxels > 0 else np.nan
        max_z = np.nanmax(wm_data[roi_mask]) if n_voxels > 0 else np.nan
        roi_rows.append((roi_name, n_voxels, n_above, fraction_above, mean_z, max_z))

    roi_table = pd.DataFrame(roi_rows, columns=["roi", "n_voxels", "n_voxels_above", "fraction_above", "mean_z", "max_z"])
    selected_table = roi_table.loc[roi_table["n_voxels_above"] > 0].copy()
    selected_names = set(selected_table["roi"])

    # 保持原表达矩阵中的ROI顺序
    common_rois = [roi for roi in expr.index if roi in selected_names and roi in eeg.index and roi in eeg.columns]

    print("\n" + "=" * 70)
    print(f"WM voxel threshold: Z >= {threshold}")
    print(f"Selected ROIs: {len(common_rois)}")
    print("ROI names:")
    print(common_rois)

    if len(common_rois) < 3:
        print("ROI数量太少，无法计算相关。")
        summary_rows.append((threshold, len(common_rois), expr.shape[1], np.nan, np.nan, np.nan, np.nan, 0))
        continue

    expr_selected = expr.loc[common_rois]
    eeg_selected = eeg.loc[common_rois, common_rois]
    gene_coexpression = calculate_coexpression(expr_selected)

    r_value, p_pearson, n_edges = corr_upper_triangle(gene_coexpression, eeg_selected, min_edges=min_edges)
    r_qap, p_qap_two, p_qap_greater, valid_perm = qap_test(gene_coexpression, eeg_selected, n_perm=n_perm, seed=random_seed + threshold_index, min_edges=min_edges)

    print("Expression matrix shape:", expr_selected.shape)
    print("Gene co-expression matrix shape:", gene_coexpression.shape)
    print("EEG matrix shape:", eeg_selected.shape)
    print("Valid edges:", n_edges)
    print(f"Pearson r: {r_value:.6f}")
    print(f"Pearson P: {p_pearson:.6g}")
    print(f"QAP two-sided P: {p_qap_two:.6g}")
    print(f"QAP greater P: {p_qap_greater:.6g}")
    print(f"Valid permutations: {valid_perm}")

    # 如需查看完整矩阵，可以取消下面两行注释
    # print("\nGene co-expression matrix:\n", gene_coexpression)
    # print("\nEEG matrix:\n", eeg_selected)

    summary_rows.append((threshold, len(common_rois), expr_selected.shape[1], r_value, p_pearson, p_qap_two, p_qap_greater, n_edges))

# =========================================================
# 4) 打印汇总表
# =========================================================
summary = pd.DataFrame(summary_rows, columns=["voxel_z_threshold", "n_roi", "n_genes", "r", "p_pearson", "p_qap_two_sided", "p_qap_greater", "n_edges"])

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print(summary.to_string(index=False))

Expression shape: (64, 1563)
EEG shape: (64, 64)
WM voxel Z range: -4.39410115910007 to 11.217749772284066

WM voxel threshold: Z >= 2
Selected ROIs: 53
ROI names:
['lh.lateralorbitofrontal_1', 'lh.parstriangularis_1', 'lh.parsopercularis_1', 'lh.rostralmiddlefrontal_1', 'lh.rostralmiddlefrontal_2', 'lh.rostralmiddlefrontal_3', 'lh.superiorfrontal_1', 'lh.superiorfrontal_2', 'lh.superiorfrontal_3', 'lh.superiorfrontal_4', 'lh.caudalmiddlefrontal_1', 'lh.precentral_1', 'lh.precentral_2', 'lh.precentral_3', 'lh.precentral_4', 'lh.paracentral_1', 'lh.caudalanteriorcingulate_1', 'lh.posteriorcingulate_1', 'lh.isthmuscingulate_1', 'lh.postcentral_2', 'lh.postcentral_3', 'lh.supramarginal_1', 'lh.supramarginal_2', 'lh.superiorparietal_1', 'lh.superiorparietal_2', 'lh.superiorparietal_3', 'lh.inferiorparietal_1', 'lh.inferiorparietal_2', 'lh.precuneus_1', 'lh.precuneus_2', 'lh.cuneus_1', 'lh.pericalcarine_1', 'lh.lateraloccipital_1', 'lh.lateraloccipital_2', 'lh.lingual_1', 'lh.lingual_2', 'l